## **Setup & Load Data**

In [ ]:
# install.packages("FNN")
# install.packages("ggrepel")
# install.packages("ggallin")
library(FNN)
library(lubridate)
library(tidyverse)
library(readr)
library(stringr)
library(bigrquery)
library(parallel)
library(ggrepel)
library(ggallin)

In [ ]:
EXPORT_BUCKET = "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055"
BILLING = "wb-silky-pepper-6055"
DATA_MOUNT = "/home/jupyter/workspace/raw/vwb-aou-datasets-controlled/v8"
# CDR_STORAGE_PATH = "gs://fc-aou-datasets-controlled/v8"
# WORKSPACE_CDR = "wb-silky-artichoke-2408.C2024Q3R8"
# previous proj = "terra-vpc-sc-d3cc1fbe"

In [ ]:
# Get analysis data
system(paste0("gsutil cp ", EXPORT_BUCKET, "/data_analysis_tsh.txt ./"))
data_anal <- read_tsv("data_analysis_tsh.txt")

# Get matched cohorts
system(paste0("gsutil cp ", EXPORT_BUCKET, "/matched_cohorts_tsh.txt ./"))
knn_cohort_t <- read_tsv("matched_cohorts_tsh.txt")

# Get personalized shifts
system(paste0("gsutil cp ", EXPORT_BUCKET, "/tsh_anal_shift.txt ./"))
df <- read.table("tsh_anal_shift.txt", header = TRUE)

## **Analysis**

In [ ]:
# Definitions re-used throughout
tsh_low <- 0.4
tsh_high <- 4.0

In [ ]:
# TODO: temp filter all values to 0-5
# head(data_anal)
# head(df)
data_anal_filt <- data_anal %>%
  dplyr::group_by(person_id) %>%
  dplyr::filter(!any(tsh < 0 | tsh > 5)) %>%
  dplyr::ungroup()
dim(data_anal)
dim(data_anal_filt)
n_distinct(data_anal_filt$person_id)

#### **Measurement-Level Confusion Matrix**

TO NOTE: this preliminary version is based only on measurement thresholds, does not include ICD codes.

In [ ]:
# Think will first do this for all measures for every individual
# 1. classify each measure based on standard thresholds
# 2. classify each measure based on personalized thresholds
df_mod <- df %>%
    dplyr::mutate(
        id = as.character(format(id, scientific = FALSE, trim = TRUE))) %>%
    dplyr::mutate(
        pers_tsh_low = tsh_low - shift,
        pers_tsh_high = tsh_high - shift)

data_anal_thresh <- inner_join(
    data_anal %>% dplyr::mutate(person_id = as.character(person_id)),
    df_mod,
    by = c("person_id" = "id"))

In [ ]:
dim(data_anal_thresh)
head(data_anal_thresh)

In [ ]:
# Make confusion matrix
data_anal_thresh <- data_anal_thresh %>%
    dplyr::mutate(conf_low = case_when(
        tsh > tsh_low & tsh > pers_tsh_low ~ "tn",
        tsh > tsh_low & tsh <= pers_tsh_low ~ "fn",
        tsh <= tsh_low & tsh > pers_tsh_low ~ "fp",
        tsh <= tsh_low & tsh <= pers_tsh_low ~ "tp"
    )) %>%
    dplyr::mutate(conf_high = case_when(
        tsh < tsh_high & tsh < pers_tsh_high ~ "tn",
        tsh < tsh_high & tsh >= pers_tsh_high ~ "fn",
        tsh >= tsh_high & tsh < pers_tsh_high ~ "fp",
        tsh >= tsh_high & tsh >= pers_tsh_high ~ "tp"
    ))

In [ ]:
table(data_anal_thresh$conf_low)
table(data_anal_thresh$conf_high)

#### **Updated Individual-Level Summary**

Define true early warning: low/high personalized before standard OR never standard

Need to get each person's earliest date of:

+ low TSH
+ high TSH
+ pers low TSH
+ pers high TSH
+ normal?

In [ ]:
# Modifying code from Hayley
# Note that this removes individuals with only "normal" measurements
date_summ <- data_anal_thresh %>%
  dplyr::mutate(
    pers_low = tsh <= pers_tsh_low,
    classic_low = tsh <= tsh_low,
    pers_high = tsh >= pers_tsh_high,
    classic_high = tsh >= tsh_high
  ) %>%
  tidyr::pivot_longer(
    cols = c(pers_low, classic_low, pers_high, classic_high),
    names_to = c("type", "direction"),
    names_sep = "_",
    values_to = "flag"
  ) %>%
  dplyr::filter(flag) %>%
  dplyr::group_by(person_id, type, direction) %>%
  dplyr::summarise(first_date = min(datetime), .groups = "drop") %>%
  dplyr::mutate(date_col = paste0("tsh_", direction, "_date_", type)) %>%
  dplyr::select(person_id, date_col, first_date) %>%
  tidyr::pivot_wider(
    names_from = date_col,
    values_from = first_date)

In [ ]:
head(date_summ)

In [ ]:
# Summary
cat("Distinct individuals:", n_distinct(data_anal_thresh$person_id), "\n")
cat("Distinct individuals with only normal classic/personalized:", length(setdiff(unique(data_anal_thresh$person_id), unique(date_summ$person_id))), "\n")
cat("Distinct individuals with classic/personalized high/low TSH:", n_distinct(date_summ$person_id), "\n")

In [ ]:
indv_low <- date_summ %>%
    dplyr::filter(!(is.na(tsh_low_date_pers) & is.na(tsh_low_date_classic))) %>%
    dplyr::mutate(
        low_status = dplyr::case_when(
            !is.na(tsh_low_date_pers) & is.na(tsh_low_date_classic) ~ "pers_only",
            
            !is.na(tsh_low_date_pers) & !is.na(tsh_low_date_classic) &
                tsh_low_date_pers < tsh_low_date_classic ~ "pers_early",
            TRUE ~ "no_diff"))
dim(indv_low)

In [ ]:
indv_high <- date_summ %>%
    dplyr::filter(!(is.na(tsh_high_date_pers) & is.na(tsh_high_date_classic))) %>%
    dplyr::mutate(
        high_status = dplyr::case_when(
            !is.na(tsh_high_date_pers) & is.na(tsh_high_date_classic) ~ "pers_only",
            
            !is.na(tsh_high_date_pers) & !is.na(tsh_high_date_classic) &
                tsh_high_date_pers < tsh_high_date_classic ~ "pers_early",
            TRUE ~ "no_diff"))
dim(indv_high)

In [ ]:
# For now, not worrying about classifying the no difference further
indv_low %>%
  count(low_status) %>%
  mutate(percent = (n / sum(n))*100)

indv_high %>%
  count(high_status) %>%
  mutate(percent = (n / sum(n))*100)

#### **Percentage Left- & Right-Shifted**

In [ ]:
shift_summ <- df_mod %>%
    dplyr::select(id, shift) %>%
    dplyr::filter(!duplicated(.)) %>%
    dplyr::mutate(shift_dir = case_when(
        shift < 0 ~ "neg",
        shift > 0 ~ "pos", 
        shift == 0 ~ "none"))
dim(shift_summ)